# GeoPackage — Vuelo 17 Región I
Genera un GeoPackage con las detecciones georreferenciadas del Vuelo 17.

### Archivos que necesitas subir:
1. `detecciones_georref_completo.csv` — detecciones con proyección nadiral
2. `sync_table.csv` — telemetría sincronizada

In [ ]:
# Celda 1 — Instalar dependencias
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'geopandas', 'fiona', 'shapely', '-q'], check=True)
print('Dependencias instaladas')

In [ ]:
# Celda 2 — Subir archivos
from google.colab import files
import os

print('Sube los archivos:')
print('  1. detecciones_georref_completo.csv')
print('  2. sync_table.csv')
uploaded = files.upload()
for nombre in uploaded:
    print(f'  OK: {nombre}  ({os.path.getsize(nombre)/1024:.0f} KB)')

In [ ]:
# Celda 3 — Verificar datos
import pandas as pd
import numpy as np

df = pd.read_csv('detecciones_georref_completo.csv')
df = df.dropna(subset=['lat_criadero', 'lon_criadero']).copy()

print(f'Detecciones totales:  {len(df):,}')
print(f'Frames únicos:        {df["frame_id"].nunique():,}')
print(f'lat_criadero:         {df["lat_criadero"].min():.6f} → {df["lat_criadero"].max():.6f}')
print(f'lon_criadero:         {df["lon_criadero"].min():.6f} → {df["lon_criadero"].max():.6f}')
print()
print('Distribución por clase:')
print(df['class_name'].value_counts().to_string())

In [ ]:
# Celda 4 — Filtrar cada 300 frames
FRAME_STEP = 300
CRS_GEO    = 'EPSG:4326'
GPKG_PATH  = 'vuelo17_regionI.gpkg'

df_filtrado = df[df['frame_id'] % FRAME_STEP == 0].copy()
print(f'Detecciones filtradas (cada {FRAME_STEP} frames): {len(df_filtrado):,}')
print(df_filtrado['class_name'].value_counts().to_string())

In [ ]:
# Celda 5 — Generar capas por clase
import geopandas as gpd
from shapely.geometry import Point, LineString

cols = [c for c in [
    'frame_id', 't_video_s', 'class_name', 'confidence',
    'u', 'v', 'lat', 'lon', 'alt_m',
    'pitch_deg', 'roll_deg', 'yaw_deg', 'speed_mps',
    'lat_criadero', 'lon_criadero',
] if c in df_filtrado.columns]

for clase in sorted(df_filtrado['class_name'].unique()):
    df_clase = df_filtrado[df_filtrado['class_name'] == clase].copy()
    gdf = gpd.GeoDataFrame(
        df_clase[cols].reset_index(drop=True),
        geometry=[Point(r['lon_criadero'], r['lat_criadero'])
                  for _, r in df_clase.iterrows()],
        crs=CRS_GEO,
    )
    gdf.to_file(GPKG_PATH, layer=clase, driver='GPKG', mode='a')
    print(f'  OK {clase:<10} {len(gdf):,} puntos')

print(f'\nCapas de criaderos guardadas en: {GPKG_PATH}')

In [ ]:
# Celda 6 — Trayectoria del dron
df_sync = pd.read_csv('sync_table.csv')

# Puntos de telemetría
gdf_telem = gpd.GeoDataFrame(
    df_sync,
    geometry=[Point(r['lon'], r['lat']) for _, r in df_sync.iterrows()],
    crs=CRS_GEO,
)
gdf_telem.to_file(GPKG_PATH, layer='puntos_telemetria', driver='GPKG', mode='a')
print(f'  OK puntos_telemetria  {len(gdf_telem):,} puntos')

# Línea de trayectoria
coords = list(zip(df_sync['lon'], df_sync['lat']))
gdf_tray = gpd.GeoDataFrame(
    [{'vuelo': 'vuelo17_regionI'}],
    geometry=[LineString(coords)],
    crs=CRS_GEO,
)
gdf_tray.to_file(GPKG_PATH, layer='trayectoria', driver='GPKG', mode='a')
print(f'  OK trayectoria')

# Buffer de cobertura
try:
    CRS_UTM = 'EPSG:31983'
    buf = gdf_tray.to_crs(CRS_UTM).buffer(200)
    gpd.GeoDataFrame(geometry=buf, crs=CRS_UTM).to_file(
        GPKG_PATH, layer='cobertura_200m', driver='GPKG', mode='a')
    print(f'  OK cobertura_200m')
except Exception as e:
    print(f'  Buffer omitido: {e}')

In [ ]:
# Celda 7 — Verificar capas finales
import fiona

capas = fiona.listlayers(GPKG_PATH)
print(f'Capas en {GPKG_PATH}:')
for c in capas:
    gdf = gpd.read_file(GPKG_PATH, layer=c)
    print(f'  {c:<25} {len(gdf):,} geometrias')

In [ ]:
# Celda 8 — Descargar
files.download(GPKG_PATH)
print(f'Descargando: {GPKG_PATH}')